# Boolean synthesis: two EPFL case studies

Follow the same functions through **equations → digital gates → XAG / AIG / k-LUT → quantum circuits**.

We use two [EPFL benchmarks](https://github.com/lsils/benchmarks): **int2float** (integer-to-floating-point conversion) and **cavlc** (variable-length coding). For readable diagrams, inspect output **3** of int2float and outputs **4, 5** of cavlc, using zero-based output indices. These are benchmark-derived output cones, not the complete benchmarks. The final section compares the complete benchmarks too.

The first cone exposes XOR structure; the second emphasizes shared AND logic. Gate diversity is measured, not assumed: an XAG is allowed to contain no XOR nodes.

## 1. Setup

Use the [setup guide](../../docs/setup.md), including system Graphviz, and build the native helper explicitly. This study shares the [Boolean workspace tutorial](../tutorials/boolean_workspace.ipynb) environment.

In VS Code, select the repository’s `.venv` Python kernel and run all cells. The setup cell finds the checkout from the kernel’s working directory and uses its existing native build and `.cache/epfl` unless you have configured other paths. `LOGIQUE_WORKSPACE` can specify a checkout when the kernel starts elsewhere. Artifacts go to the checkout’s `dump/` directory.


In [ ]:
from pathlib import Path
import os
import shutil
from uuid import uuid4
from logique.paths import native_executable, cache_directory
from logique.benchmarks.runner import new_run_directory

# Editor kernels may start in experiments/studies and lack terminal exports.
if "LOGIQUE_WORKSPACE" in os.environ:
    ROOT = Path(os.environ["LOGIQUE_WORKSPACE"]).expanduser().resolve()
else:
    current = Path.cwd().resolve()
    ROOT = next(
        (
            path
            for path in (current, *current.parents)
            if (path / "pyproject.toml").is_file()
            and (path / "external/native/boolean_synthesis").is_dir()
        ),
        current,
    )

# Preserve explicit settings and helpers installed on PATH.
if "LOGIQUE_NATIVE_EXECUTABLE" not in os.environ and not shutil.which(
    "boolean_synthesis"
):
    helper = ROOT / "external/native/boolean_synthesis/build/boolean_synthesis"
    if helper.is_file():
        os.environ["LOGIQUE_NATIVE_EXECUTABLE"] = str(helper)
if "LOGIQUE_CACHE" not in os.environ and (ROOT / ".cache/epfl").is_dir():
    os.environ["LOGIQUE_CACHE"] = str(ROOT / ".cache/epfl")
native_executable()  # Build explicitly during setup; never compile in the notebook.

OUT = (
    Path(os.environ["LOGIQUE_OUTPUT"]).expanduser().resolve()
    if "LOGIQUE_OUTPUT" in os.environ
    else new_run_directory(ROOT / "dump" / f"boolean-visual-{uuid4().hex}")
)
OFFLINE = os.environ.get("LOGIQUE_OFFLINE", "0") == "1"


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Math, SVG, HTML
from logique.synthesis import synthesize, METHODS
from logique.benchmarks.datasets import download_benchmarks, benchmark_manifest
from logique.benchmarks.provenance import environment
from logique.benchmarks.export import export_result
from logique.verification import validate, truth_outputs, verify_small_quantum
from logique.verification.classical import input_assignments
from logique.workspace import workspace_analysis, borrowing_demo, verify_borrowing
from logique.visualization.circuits import network_figure, circuit_window
from logique.visualization import boolean as bv
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

from io import StringIO
assert shutil.which("dot"), "Install system Graphviz."
paths = download_benchmarks(("int2float", "cavlc"), offline=OFFLINE)

selected = {"int2float": [3], "cavlc": [4, 5]}
results = {name: {method: synthesize(paths[name], method, outputs=outputs, k=4) for method in METHODS} for name, outputs in selected.items()}
for methods in results.values():
    for result in methods.values():
        validate(result)
display(pd.DataFrame([{ "benchmark": name, "original outputs": outputs, **bv.graph_stats(results[name]["xag"].metadata["network"])} for name, outputs in selected.items()]))


## 2. Boolean functions in LaTeX

These are **exact factored definitions**, not expanded sum-of-products. Read top to bottom: each intermediate `t` is defined once. Input $x_i$ is primary input $i$ in the imported AIGER file, including unused inputs. For int2float, $y_0$ is original output 3; for cavlc, $(y_0,y_1)$ are original outputs (4,5).

$\land$ means AND, $\lor$ means OR, $\oplus$ means XOR, and a bar means NOT. De Morgan notation exposes NOR gates; it does not change the function or native graph. Signal numbers are local to each representation.

In [ ]:
for name in selected:
    graph = results[name]["xag"].metadata["network"]
    display(Markdown(f"### {name}: outputs {selected[name]}"))
    for block in bv.latex_blocks(graph):
        display(Math(block))
    assignments = input_assignments(len(graph["inputs"]))
    np.testing.assert_array_equal(
        bv.evaluate_gate_rows(graph, assignments),
        truth_outputs(graph, assignments),
    )
    (OUT / f"{name}_selected.tex").write_text(
        "\n\n".join("$$" + block + "$$" for block in bv.latex_blocks(graph))
    )

## 3. Digital logic with Schemdraw

Each [Schemdraw circuit](https://schemdraw.readthedocs.io/en/stable/elements/logic.html) is fully wired from its used primary inputs to its outputs. Gate labels match the intermediate signals in the equations. **Filled dots are connected fanouts; crossings with a gap are not connected.** Hollow gate bubbles mean inversion. Shared gates are drawn only once, with wires to every consumer. Graphviz routes the wires; Schemdraw renders the actual logic symbols and connections. Unused primary inputs are omitted from the drawing only. Scroll the vector drawings to inspect them at full size.

In [ ]:
for name in selected:
    display(Markdown(f"### {name}"))
    fig = bv.schematic_figure(results[name]["xag"].metadata["network"])
    fig.savefig(OUT / f"{name}_logic.png", dpi=150, bbox_inches="tight")
    buffer = StringIO()
    fig.savefig(buffer, format="svg", bbox_inches="tight")
    svg = buffer.getvalue()
    (OUT / f"{name}_logic.svg").write_text(svg)
    display(HTML('<div style="overflow:auto;max-height:750px;border:1px solid #ddd">'
                 + SVG(svg).data + '</div>'))
    plt.close(fig)

## 4. XAG, AIG, and 4-LUT networks

- **AIG:** two-input AND nodes with complemented edges; even XOR functions must be expressed using this basis.
- **XAG:** adds two-input XOR nodes. XOR-rich functions may become smaller, but the result depends on factoring.
- **k-LUT:** covers logic with lookup tables of at most $k$ inputs. Each LUT can implement an arbitrary local Boolean function; fewer nodes do not necessarily mean fewer quantum gates.

These are actual exported mockturtle networks. AIG keeps the imported AND structure; XAG applies the bridge's refactoring, and LUT mapping starts from that XAG. This comparison includes both representation and preprocessing effects, not a controlled optimizer-quality contest.

[Graphviz dot](https://graphviz.org/docs/layouts/dot/) gives layered, connected **SVG** graphs that stay sharp when zoomed. Green = AND; amber = XOR; blue = LUT. Red dashed edges with circles mean NOT. Numbered ports specify fanin order. Unused inputs are hidden only in the drawing. For LUT truth tables, port 0 is the least-significant assignment bit; the displayed bit string runs from the highest assignment to zero.

In [ ]:
representations = {"XAG": "xag", "AIG": "aig_bennett", "4-LUT": "klut_bennett"}

def show_vector(dot, filename):
    dot.save(str(OUT / f"{filename}.dot"))
    svg = dot.pipe(format="svg").decode()
    (OUT / f"{filename}.svg").write_text(svg)
    # Scroll at the SVG's natural size instead of shrinking node labels.
    display(HTML('<div style="overflow:auto;max-height:650px;border:1px solid #ddd">'
                 + SVG(svg).data + '</div>'))

for name in selected:
    display(Markdown(f"### {name}"))
    display(pd.DataFrame([
        {"representation": label, **bv.graph_stats(results[name][method].metadata["network"])}
        for label, method in representations.items()
    ]))
    reference = results[name]["xag"].metadata["network"]
    assignments = input_assignments(len(reference["inputs"]))
    for label, method in representations.items():
        graph = results[name][method].metadata["network"]
        np.testing.assert_array_equal(
            truth_outputs(reference, assignments), truth_outputs(graph, assignments)
        )
        show_vector(bv.graphviz_graph(graph, f"{name}: {label}"),
                    f"{name}_{label.lower()}_selected")
        if label == "4-LUT":
            display(pd.DataFrame(bv.lut_table(graph)))

### Output selection changes the graph

The output cones are extracted **before** refactoring. Joint output sharing can change which XOR patterns the heuristic exposes. An AND-only XAG does not prove that the Boolean function has no useful XOR decomposition.

In [ ]:
scope_rows = []
for name, outputs in selected.items():
    for scope in [[o] for o in outputs] + ([outputs] if len(outputs) > 1 else []):
        r = synthesize(paths[name], "xag", outputs=scope)
        scope_rows.append({"benchmark": name, "outputs": str(scope),
                           **bv.graph_stats(r.metadata["network"])})
display(pd.DataFrame(scope_rows))

## 5. Five quantum compilation methods

These labels match the existing tutorial; they are not five independent algorithms:

| Method | What it does |
|---|---|
| XAG specialized | Uses XOR-aware synthesis and temporary AND computation with phase-sensitive controlled rotations and cleanup. |
| AIG + Bennett | Computes AND nodes into workspace, retains intermediates, copies outputs, then reverses computation. |
| XAG + Bennett | The same retain-and-reverse schedule on the refactored AND/XOR graph. |
| 4-LUT + Bennett | Computes mapped LUT functions reversibly, then reverses intermediates. LUT arity bounds the local classical function, not the total qubit count. |
| Best-fit LHRS | Uses LUT-based hierarchical reversible synthesis with a best-fit mapping strategy and earlier cleanup/reuse. Its local LUT synthesis can cost more gates while using less workspace. |

**Bennett is an uncomputation schedule**, not a graph representation. Best-fit LHRS uses outer cut 16 and inner cut 4 here; it is not identical to fixed 4-LUT mapping.

Below are the **complete raw synthesis circuits**, not truncated windows. Inputs and workspace start in the documented computational/zero states; raw output wires may alias inputs. Use `result.oracle()` for the uniform contract $|x\rangle|y\rangle|0\rangle \mapsto |x\rangle|y\oplus f(x)\rangle|0\rangle$. That wrapper can add qubits and gates.

Controlled $R_x(\pm\pi)$ is not an exact Toffoli: keep its phase. Counts below are logical Qiskit operations, not hardware-decomposed costs.

In [ ]:
columns = ["benchmark", "method", "qubits", "workspace", "gates", "depth",
           "exact_ccx", "controlled_rx_pi", "mcx_3plus", "gate_types", "validation"]
comparison = pd.DataFrame([
    {"benchmark": name, **r.summary()}
    for name, methods in results.items() for r in methods.values()
])
display(comparison[columns])

def show_circuit(result, filename):
    fig = result.circuit.draw("mpl", idle_wires=False, fold=-1)
    buffer = StringIO()
    fig.savefig(buffer, format="svg", bbox_inches="tight")
    plt.close(fig)
    svg = buffer.getvalue()
    (OUT / f"{filename}.svg").write_text(svg)
    display(HTML('<div style="overflow:auto;max-height:650px;border:1px solid #ddd">'
                 + SVG(svg).data + '</div>'))

for name, methods in results.items():
    for method, result in methods.items():
        display(Markdown(f"### {name}: {method}"))
        display(pd.DataFrame({
            "role": ["inputs", "outputs"],
            "raw qubit indices": [result.metadata["input_qubits"], result.metadata["output_qubits"]]
        }))
        show_circuit(result, f"{name}_{method}_circuit")
        export_result(result, OUT, f"{name}_{method}_selected")

### Vary LUT size

Changing $k$ trades local synthesis difficulty against network size and workspace. Compare logical gates and qubits together; a large controlled gate still counts as one operation here.

In [ ]:
lut_sizes = []
for name, outputs in selected.items():
    for k in (3, 4, 6):
        r = synthesize(paths[name], "klut_bennett", outputs=outputs, k=k)
        validate(r)
        lut_sizes.append({"benchmark": name, **r.summary()})
display(pd.DataFrame(lut_sizes)[["benchmark", "k", "LUT", "qubits", "workspace",
                                "gates", "depth", "mcx_3plus"]])

## 6. Workspace behavior

Count workspace wires that are nonzero for **at least one** input at each expanded gate boundary. This is not the Hamming weight of a quantum state. The candidate table lists conditional known-value facts verified over all original inputs.

A fact such as “under $q_t=1$, $q_c=1$” is a starting point for conditional workspace, **not permission to insert arbitrary gates**. Any inserted computation must respect the protecting condition, restore borrowed wires, and preserve phases before the original circuit resumes. See the verified borrowing example in the [original notebook](../tutorials/boolean_workspace.ipynb) and [conditionally clean ancilla paper](https://arxiv.org/html/2407.17966v2). No borrowing transformation is applied here.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6), constrained_layout=True)
facts = []
for ax, (name, methods) in zip(axes, results.items()):
    for method, result in methods.items():
        report = workspace_analysis(result, expanded=True)
        ax.step(range(len(report["occupancy"])), report["occupancy"],
                where="post", label=method)
        facts.extend({"benchmark": name, "method": method, **c}
                     for c in report["candidates"])
    ax.set(title=name, xlabel="Expanded gate index", ylabel="Potentially nonzero workspace")
    ax.legend(fontsize=8)
display(fig)
fig.savefig(OUT / "workspace.png", dpi=150)
plt.close(fig)
facts = pd.DataFrame(facts)
display(facts.groupby(["benchmark", "method"]).size().rename("candidate facts").to_frame())
display(facts.head(12))
facts.to_csv(OUT / "conditional_facts.csv", index=False)

## 7. Full-benchmark context

Now compile **every output** of both benchmarks with all five methods. These numbers are separate from the readable cones above. Export full graph SVG/DOT files, exact factored LaTeX, and raw/wrapped QPY circuits for deeper inspection; large diagrams are not squeezed into the notebook.

All functions have at most 11 inputs, so validation is exhaustive. The verifier tracks the circuit permutation **and complex phases** on every input basis state, checks native-to-Qiskit agreement, and checks wrapped oracle restoration. By linearity this covers superposed inputs for these monomial circuits with initially clean workspace; it is not a dense statevector simulation or a hardware-noise test.

In [ ]:
full_results = {}
full_rows = []
for name in selected:
    full_results[name] = {}
    for method in METHODS:
        r = synthesize(paths[name], method, k=4)
        validate(r)
        full_results[name][method] = r
        full_rows.append({"benchmark": name, **r.summary()})
        export_result(r, OUT, f"{name}_{method}_full")
    graph = full_results[name]["xag"].metadata["network"]
    (OUT / f"{name}_full.tex").write_text(
        "\n\n".join("$$" + block + "$$" for block in bv.latex_blocks(graph))
    )
    for label, method in representations.items():
        graph = full_results[name][method].metadata["network"]
        dot = bv.graphviz_graph(graph, f"{name}: full {label}")
        dot.save(str(OUT / f"{name}_{label.lower()}_full.dot"))
        (OUT / f"{name}_{label.lower()}_full.svg").write_bytes(dot.pipe(format="svg"))
full_comparison = pd.DataFrame(full_rows)
display(full_comparison[columns])
comparison.to_csv(OUT / "selected_comparison.csv", index=False)
full_comparison.to_csv(OUT / "full_comparison.csv", index=False)
print(f"Saved reproducible artifacts to {OUT}")

## 8. Observations and next checks

With these pinned inputs and settings:

- int2float's selected XAG has 8 ANDs and 4 XORs, versus 13 ANDs in the AIG. Specialized XAG uses 7 workspace qubits and 32 logical operations; XAG/Bennett uses 11 and 32. The operations have different phase semantics, so equal counts are not equal hardware costs.
- Best-fit uses no extra workspace for either cone, but int2float grows to 257 logical operations, compared with 39 for 4-LUT/Bennett. This is a concrete space/gate tradeoff, not a universal ranking.
- The joint CAVLC cone has 19 ANDs, no explicit XORs, and 5 shared internal nodes. AIG/Bennett and XAG/Bennett consequently give the same counts here.

1. Locate XORs in int2float's equations, then follow their CNOT-based computation in the XAG circuits.
2. Follow shared CAVLC nodes through graph fanout and the retained/reused quantum workspace.
3. Compare the same XAG under specialized synthesis and Bennett before attributing differences to the graph representation.
4. Compare LUT ports and truth tables with their expanded controlled gates. Smaller classical graphs need not yield cheaper quantum circuits.
5. Inspect conditional facts at their exact gate boundaries. Candidate counts alone do not measure useful borrowable capacity.

Re-run from the repository root:
```bash
uv run --no-sync logique notebook run experiments/studies/boolean_visual_study.ipynb
```
Add `--offline` once the native build and benchmark cache exist. Generated artifacts stay under the ignored results directory.